# Lab 04 — Silver data quality and quarantine

This notebook converts the immutable Bronze records into a controlled Silver-ready candidate batch. It applies explicit business-quality rules, attaches all applicable rejection reasons, quarantines invalid records, and removes exact duplicates from the valid population.

## Objectives

- select one Bronze batch using `_batch_id`;
- evaluate every row against named quality rules;
- retain the first occurrence of an exact business duplicate and reject later occurrences;
- persist rejected rows and their reason arrays in a Delta quarantine table;
- normalize valid records into an explicit Silver candidate schema;
- write batch-level quality metrics idempotently;
- prove that total rows equal valid rows plus rejected rows.

> This notebook does **not** merge into the final Silver table. That responsibility belongs to `lab04_04_silver_merge.ipynb`.

## 1. Load shared configuration

In [0]:
%run ./lab04_00_config

# Lab 04 — Runtime Configuration

This notebook is intentionally **DDL-free**.

It:
- defines runtime widgets and validates their values;
- builds reusable paths and table names;
- loads the version-controlled YAML data contracts;
- exposes one runtime-selected contract plus the v1/v2 references required by the schema-governance demonstrations.

It does **not** create catalogs, schemas, volumes, folders, or tables.

Run `lab04_00_setup` manually once when the Lab 4 structure must be created or verified. Production Job tasks may `%run ./lab04_00_config` safely.


runtime_selection,contract_name,yaml_version,governance_status,column_count,supersedes,runtime_selected
v1,online_retail,1,active,8,null,true


Runtime configuration ready: dbr_dev.parvinbadalov
Volume: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality
Source workbook: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/Online Retail.xlsx
Runtime contract: online_retail v1 (governance status: active)
Schema policy: fail


In [0]:
import sys

from delta.tables import DeltaTable
from pyspark.sql import functions as F

lab04_root = (
    "/Workspace/Users/parvinbadalov@yahoo.com/"
    "Databricks-Academy-Lakehouse/labs/lab_04_silver_quality"
)

if lab04_root not in sys.path:
    sys.path.append(lab04_root)

from src.quality_rules import (
    EXPECTED_SOURCE_COLUMNS,
    apply_online_retail_quality_rules,
    build_quality_metrics,
    build_rule_failure_summary,
    split_valid_and_quarantine,
)

# Keep config and reusable module aligned.
if tuple(expected_source_columns) != tuple(EXPECTED_SOURCE_COLUMNS):
    raise AssertionError(
        "lab04_00_config expected_source_columns does not match "
        "src.quality_rules.EXPECTED_SOURCE_COLUMNS."
    )

silver_candidate_path = (
    f"{paths['landing']}/silver_candidates/batch_id={batch_id}"
)

print(f"Bronze source: {bronze_table}")
print(f"Selected batch: {batch_id}")
print(f"Contract version: {contract_version}")
print(f"Candidate path: {silver_candidate_path}")
print(f"Quarantine table: {quarantine_table}")
print(f"Quality metrics table: {quality_metrics_table}")
print("Reusable quality module: src.quality_rules")


Bronze source: dbr_dev.parvinbadalov.lab04_bronze_retail
Selected batch: initial
Contract version: v1
Candidate path: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/landing/silver_candidates/batch_id=initial
Quarantine table: dbr_dev.parvinbadalov.lab04_quarantine
Quality metrics table: dbr_dev.parvinbadalov.lab04_quality_metrics
Reusable quality module: src.quality_rules


## 2. Read and validate the selected Bronze batch

The batch filter prevents an incremental run from re-evaluating unrelated records. Before applying business rules, the notebook verifies that the Bronze table exists and contains the expected source and technical columns.

In [0]:
if not spark.catalog.tableExists(bronze_table):
    raise FileNotFoundError(
        f"Bronze table {bronze_table} does not exist. "
        "Run lab04_02_bronze_ingestion.ipynb first."
    )

bronze_df = spark.table(bronze_table)
required_bronze_columns = set(expected_source_columns) | {
    "_bronze_record_id",
    "_batch_id",
    "_record_hash",
    "_source_row_number",
    "_source_file",
    "_source_sheet",
    "_input_file_path",
    "_bronze_ingested_at",
}
missing_columns = sorted(required_bronze_columns - set(bronze_df.columns))
if missing_columns:
    raise AssertionError(f"Bronze is missing required columns: {missing_columns}")

batch_bronze_df = bronze_df.filter(F.col("_batch_id") == F.lit(batch_id))
bronze_batch_count = batch_bronze_df.count()
if bronze_batch_count == 0:
    available_batches = [row["_batch_id"] for row in bronze_df.select("_batch_id").distinct().collect()]
    raise ValueError(
        f"No Bronze rows found for batch_id={batch_id!r}. "
        f"Available batches: {sorted(available_batches)}"
    )

print(f"Bronze rows selected: {bronze_batch_count:,}")
display(batch_bronze_df.limit(20))

Bronze rows selected: 433,737


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,_source_row_number,_source_file,_source_sheet,_prepared_at_utc,_record_hash,_input_file_path,_input_file_name,_input_file_size,_input_file_modified_at,_bronze_record_id,_batch_id,_source_system,_contract_version,_bronze_ingested_at,_bronze_ingestion_date
536367,22622,BOX OF VINTAGE ALPHABET BLOCKS,2,2010-12-01T08:34:00.000,9.95,13047,United Kingdom,18,Online Retail.xlsx,Online Retail,2026-08-09T20:24:07.431Z,52b3903bdfddf7c75bb40dfff03108731e093063fdda8e5907b34d52c6aa3cc4,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00000-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-307-1.c000.snappy.parquet,part-00000-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-307-1.c000.snappy.parquet,2366938,2026-08-09T20:24:09.000Z,f05ad52e5190405cfb896f60c3db66a0cbadd1170b1943cfee5a9e0ad0f03761,initial,uci_online_retail,v1,2026-08-09T20:37:30.749Z,2026-08-09
536370,22326,ROUND SNACK BOXES SET OF4 WOODLAND,24,2010-12-01T08:45:00.000,2.95,12583,France,36,Online Retail.xlsx,Online Retail,2026-08-09T20:24:07.431Z,34611458c2920721d347f0b7e4624c4b3014a6faf1e0fe2bf8eddae016bb3c9a,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00000-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-307-1.c000.snappy.parquet,part-00000-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-307-1.c000.snappy.parquet,2366938,2026-08-09T20:24:09.000Z,52fc3eaea6c815368d9598ea72ab18675945143275a399954d89d223225ac5f9,initial,uci_online_retail,v1,2026-08-09T20:37:30.749Z,2026-08-09
536373,21068,VINTAGE BILLBOARD LOVE/HATE MUG,6,2010-12-01T09:02:00.000,1.06,17850,United Kingdom,58,Online Retail.xlsx,Online Retail,2026-08-09T20:24:07.431Z,572ea091132d2260c40de6a1a8ea62028484e9d6f9aa0e398debaf527d44b1b7,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00000-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-307-1.c000.snappy.parquet,part-00000-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-307-1.c000.snappy.parquet,2366938,2026-08-09T20:24:09.000Z,74b0465977b50148a4df03d32b33ceccd8cdbf8f64d0902daa300ff6c76930ae,initial,uci_online_retail,v1,2026-08-09T20:37:30.749Z,2026-08-09
536375,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01T09:32:00.000,4.25,17850,United Kingdom,83,Online Retail.xlsx,Online Retail,2026-08-09T20:24:07.431Z,3f12de5c5271a40eb3bcfecefa5daefe1dc5f211df34fcbcce7479e3c744c963,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00000-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-307-1.c000.snappy.parquet,part-00000-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-307-1.c000.snappy.parquet,2366938,2026-08-09T20:24:09.000Z,343101aa6ef06ab910f612fd6e7c62e814ee4b721d486081d386a9bbda759e64,initial,uci_online_retail,v1,2026-08-09T20:37:30.749Z,2026-08-09
536378,22386,JUMBO BAG PINK POLKADOT,10,2010-12-01T09:37:00.000,1.95,14688,United Kingdom,88,Online Retail.xlsx,Online Retail,2026-08-09T20:24:07.431Z,6b5334f009e61df51caff6cdecfe106d2639f3258a1f7190d2d371c93e06773e,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00000-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-307-1.c000.snappy.parquet,part-00000-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-307-1.c000.snappy.parquet,2366938,2026-08-09T20:24:09.000Z,34de60740d53f87fbbd17fb75475168509b1ec3e064fa28f850902792f8287b5,initial,uci_online_retail,v1,2026-08-09T20:37:30.749Z,2026-08-09
536381,22083,PAPER CHAIN KIT RETROSPOT,1,2010-12-01T09:41:00.000,2.95,15311,United Kingdom,125,Online Retail.xlsx,Online Retail,2026-08-09T20:24:07.431Z,aa2e5cbc0d58346184a0ba6834aa4c68305e36c2b7a10010a89370aaaf89edb8,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00000-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-307-1.c000.snappy.parquet,part-00000-tid-76298

## 3. Apply reusable quality rules and identify duplicates

The data-quality implementation lives in `src/quality_rules.py`, so notebooks, Jobs, and tests use one shared rule definition.

Exact business duplicates are identified across the eight Online Retail source fields. The first deterministic occurrence is retained; later occurrences receive `DUPLICATE_BUSINESS_ROW`. Technical metadata is excluded from duplicate matching.


In [0]:
quality_df = apply_online_retail_quality_rules(
    batch_bronze_df,
    contract_version=contract_version,
    business_columns=EXPECTED_SOURCE_COLUMNS,
)

duplicate_row_count = quality_df.filter(
    F.col("_duplicate_rank") > 1
).count()

print(f"Exact duplicate rows to reject: {duplicate_row_count:,}")


Exact duplicate rows to reject: 3,427


## 4. Inspect named data-quality results

`src/quality_rules.py` attaches all applicable reason codes to `_quality_reasons`. A row can fail multiple checks, so the full array is retained.

| Rule code | Condition |
|---|---|
| `MISSING_INVOICE_NO` | Invoice number is null or blank |
| `MISSING_STOCK_CODE` | Product code is null or blank |
| `MISSING_DESCRIPTION` | Description is null or blank |
| `NON_POSITIVE_QUANTITY` | Quantity is null, zero, or negative |
| `MISSING_INVOICE_DATE` | Invoice timestamp is null |
| `NON_POSITIVE_UNIT_PRICE` | Price is null, zero, or negative |
| `MISSING_CUSTOMER_ID` | Customer ID is null or blank |
| `MISSING_COUNTRY` | Country is null or blank |
| `CANCELLED_INVOICE` | Invoice number begins with `C` |
| `DUPLICATE_BUSINESS_ROW` | Exact business duplicate after the first occurrence |


In [0]:
display(
    quality_df
    .groupBy("_quality_status")
    .agg(F.count("*").alias("rows"))
    .orderBy("_quality_status")
)

print(
    "✅ Quality rules applied through "
    "src.quality_rules.apply_online_retail_quality_rules()."
)


_quality_status,rows
REJECTED,118636
VALID,315101


✅ Quality rules applied through src.quality_rules.apply_online_retail_quality_rules().


## 5. Inspect rule-level failures

Exploding the reason array produces one row per rule violation. Because one record can fail multiple rules, the sum of violations can be greater than the rejected-row count.

In [0]:
rule_failure_summary_df = build_rule_failure_summary(
    quality_df
)

display(rule_failure_summary_df)

display(
    quality_df
    .filter(F.col("_quality_status") == "REJECTED")
    .select(
        *EXPECTED_SOURCE_COLUMNS,
        "_quality_reasons",
        "_bronze_record_id",
    )
    .limit(50)
)


quality_rule,failed_rows
MISSING_CUSTOMER_ID,108107
NON_POSITIVE_QUANTITY,8519
CANCELLED_INVOICE,7430
DUPLICATE_BUSINESS_ROW,3427
NON_POSITIVE_UNIT_PRICE,2026
MISSING_DESCRIPTION,1161


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,_quality_reasons,_bronze_record_id
536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01T11:45:00.000,2.95,17908,United Kingdom,List(DUPLICATE_BUSINESS_ROW),2f853e61c708cbb9e1763ac81a8a334388f00025c99bb906998c8d11449619c3
536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01T11:49:00.000,1.65,17920,United Kingdom,List(DUPLICATE_BUSINESS_ROW),d7ccd118aad8332bdfd16820c60101e672d41faa1e5449fb63bbc0157e212fe7
536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01T11:49:00.000,1.65,17920,United Kingdom,List(DUPLICATE_BUSINESS_ROW),eeec05b3695bf9350aec3bec516e6eac2c0c6820dc0a8d8ea9c85f907a429db7
536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01T11:49:00.000,1.65,17920,United Kingdom,List(DUPLICATE_BUSINESS_ROW),408de85c68436a040cf858b0071a742783119962865947c3087f6024f374658a
536412,21706,FOLDING UMBRELLA RED/WHITE POLKADOT,1,2010-12-01T11:49:00.000,4.95,17920,United Kingdom,List(DUPLICATE_BUSINESS_ROW),31784cf841704a817278df1479fa5094b2c01e3a5d62fbef03d06c0b259e6d56
536412,21708,FOLDING UMBRELLA CREAM POLKADOT,1,2010-12-01T11:49:00.000,4.95,17920,United Kingdom,List(DUPLICATE_BUSINESS_ROW),07a3e11750b588743fca44d696a16f66dd97e90b59397cbcda50a5a421c51b28
536412,22273,FELTCRAFT DOLL MOLLY,1,2010-12-01T11:49:00.000,2.95,17920,United Kingdom,List(DUPLICATE_BUSINESS_ROW),db4e34aced71db0ed17c7643efd6dadc78b4c8dc8fc861afa7ff97045a0e500a
536412,22749,FELTCRAFT PRINCESS CHARLOTTE DOLL,1,2010-12-01T11:49:00.000,3.75,17920,United Kingdom,List(DUPLICATE_BUSINESS_ROW),6de1938895e7b30433fdb0938611bc896390ed4dd081924534211dab512efb9b
536412,22900,SET 2 TEA TOWELS I LOVE LONDON,2,2010-12-01T11:49:00.000,2.95,17920,United Kingdom,List(DUPLICATE_BUSINESS_ROW),995f9cb804506841a5e6dac1b4d3cc0a05f03504b5b564c3ad0226d49f24a68a
536412,22902,TOTE BAG I LOVE LONDON,7,2010-12-01T11:49:00.000,2.1,17920,United Kingdom,List(DUPLICATE_BUSINESS_ROW),72b9930b80f3155b6580eaddd6fd8515add49c256f621020aadaaa8a8db78c96


## 6. Split valid and rejected records

The split is exhaustive and mutually exclusive. A reconciliation assertion ensures that no Bronze row is lost or counted twice.

In [0]:
valid_quality_df, rejected_quality_df = (
    split_valid_and_quarantine(quality_df)
)

valid_count = valid_quality_df.count()
rejected_count = rejected_quality_df.count()

if valid_count + rejected_count != bronze_batch_count:
    raise AssertionError(
        f"Quality reconciliation failed: valid={valid_count}, "
        f"rejected={rejected_count}, Bronze={bronze_batch_count}."
    )

print(f"Bronze rows: {bronze_batch_count:,}")
print(f"Valid rows: {valid_count:,}")
print(f"Rejected rows: {rejected_count:,}")
print("✅ Reconciliation passed: Bronze = valid + rejected.")


Bronze rows: 433,737
Valid rows: 315,101
Rejected rows: 118,636
✅ Reconciliation passed: Bronze = valid + rejected.


## 7. Persist rejected rows to the pre-created quarantine table

The quarantine table structure is created by `lab04_00_setup` outside the production Job.

Rejected rows are upserted by a stable quarantine key so retries remain idempotent. The Job notebook validates the target but performs no structural DDL.


In [0]:
quarantine_batch_df = (
    rejected_quality_df
    .withColumn(
        "_quarantine_record_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("_bronze_record_id"),
                F.lit(contract_version),
            ),
            256,
        ),
    )
    .withColumn("_quarantined_at", F.current_timestamp())
)

if reset_demo_objects:
    raise ValueError(
        "reset_demo_objects=true is not allowed inside the production Job. "
        "Run lab04_00_setup manually when a clean structural rebuild is required."
    )

if not spark.catalog.tableExists(quarantine_table):
    raise RuntimeError(
        f"Required quarantine target does not exist: {quarantine_table}. "
        "Run lab04_00_setup manually before executing the Job."
    )

target_columns = set(spark.table(quarantine_table).columns)
incoming_columns = set(quarantine_batch_df.columns)

if target_columns != incoming_columns:
    raise AssertionError(
        "Quarantine target schema does not match the quality output. "
        f"Missing in target: {sorted(incoming_columns - target_columns)}; "
        f"extra in target: {sorted(target_columns - incoming_columns)}"
    )

(
    DeltaTable.forName(spark, quarantine_table)
    .alias("target")
    .merge(
        quarantine_batch_df.alias("source"),
        "target._quarantine_record_id = source._quarantine_record_id",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

quarantine_batch_count = (
    spark.table(quarantine_table)
    .filter(
        (F.col("_batch_id") == batch_id)
        & (F.col("_quality_contract_version") == contract_version)
    )
    .count()
)

if quarantine_batch_count != rejected_count:
    raise AssertionError(
        f"Quarantine count mismatch: expected {rejected_count}, "
        f"found {quarantine_batch_count}."
    )

print(
    f"✅ Quarantine persisted idempotently: "
    f"{quarantine_batch_count:,} rows for this batch and contract."
)


✅ Quarantine persisted idempotently: 118,636 rows for this batch and contract.


## 8. Normalize the valid Silver candidate schema

Only records that passed every rule reach this step. Column names are converted to a consistent `snake_case` contract, text is trimmed, numeric types are explicit, and analytics columns are derived. The original Bronze identity and lineage remain available for traceability.

In [0]:
silver_candidate_df = (
    valid_quality_df
    .select(
        F.col("_bronze_record_id").alias("transaction_line_id"),
        F.trim(F.col("InvoiceNo")).alias("invoice_no"),
        F.trim(F.col("StockCode")).alias("stock_code"),
        F.trim(F.col("Description")).alias("description"),
        F.col("Quantity").cast("long").alias("quantity"),
        F.col("InvoiceDate").cast("timestamp").alias("invoice_timestamp"),
        F.col("UnitPrice").cast("decimal(18,4)").alias("unit_price"),
        F.trim(F.col("CustomerID")).alias("customer_id"),
        F.trim(F.col("Country")).alias("country"),
        F.col("_record_hash").alias("source_record_hash"),
        F.col("_batch_id").alias("source_batch_id"),
        F.col("_source_file").alias("source_file"),
        F.col("_source_sheet").alias("source_sheet"),
        F.col("_source_row_number").alias("source_row_number"),
        F.col("_input_file_path").alias("input_file_path"),
        F.col("_bronze_ingested_at").alias("bronze_ingested_at"),
        F.col("_quality_contract_version").alias("quality_contract_version"),
        F.col("_quality_checked_at").alias("quality_checked_at"),
    )
    .withColumn("sales_amount", (F.col("quantity") * F.col("unit_price")).cast("decimal(20,4)"))
    .withColumn("invoice_date", F.to_date("invoice_timestamp"))
    .withColumn("invoice_year", F.year("invoice_timestamp"))
    .withColumn("invoice_month", F.month("invoice_timestamp"))
    .withColumn("silver_prepared_at", F.current_timestamp())
)

candidate_count = silver_candidate_df.count()
candidate_distinct_ids = silver_candidate_df.select("transaction_line_id").distinct().count()
if candidate_count != valid_count or candidate_count != candidate_distinct_ids:
    raise AssertionError(
        f"Silver candidate uniqueness failed: rows={candidate_count}, "
        f"valid={valid_count}, distinct IDs={candidate_distinct_ids}."
    )

silver_candidate_df.printSchema()
display(silver_candidate_df.limit(20))

root
 |-- transaction_line_id: string (nullable = true)
 |-- invoice_no: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- invoice_timestamp: timestamp (nullable = true)
 |-- unit_price: decimal(18,4) (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- source_record_hash: string (nullable = true)
 |-- source_batch_id: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- source_sheet: string (nullable = true)
 |-- source_row_number: long (nullable = true)
 |-- input_file_path: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- quality_contract_version: string (nullable = false)
 |-- quality_checked_at: timestamp (nullable = false)
 |-- sales_amount: decimal(20,4) (nullable = true)
 |-- invoice_date: date (nullable = true)
 |-- invoice_year: integer (nullable = true)
 |-- invoice_mon

transaction_line_id,invoice_no,stock_code,description,quantity,invoice_timestamp,unit_price,customer_id,country,source_record_hash,source_batch_id,source_file,source_sheet,source_row_number,input_file_path,bronze_ingested_at,quality_contract_version,quality_checked_at,sales_amount,invoice_date,invoice_year,invoice_month,silver_prepared_at
6c4de07354ac2a12840cb7ab18e29b9d0299b88486a9bd6288b25d7347f167f2,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01T08:26:00.000Z,4.2500,17850,United Kingdom,d437e01828afac453bca294ee923cdc7ae1985b362293d1004075e5b0cc14b40,initial,Online Retail.xlsx,Online Retail,8,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00004-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-313-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-10T20:53:49.431Z,25.5000,2010-12-01,2010,12,2026-08-10T20:53:49.431Z
7d07613aeedc44e44ab851ad437283d55529b57cf01ca858f2db461bb85de370,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01T08:26:00.000Z,7.6500,17850,United Kingdom,0d605467e0138f6396f9c149ce5ad5ab7096c7056518578056ef5dc2dc38dfbd,initial,Online Retail.xlsx,Online Retail,7,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00005-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-314-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-10T20:53:49.431Z,15.3000,2010-12-01,2010,12,2026-08-10T20:53:49.431Z
17000e568d97768ddfb50b30790fb358289f6cd675f7925ce9ada92655c8f70b,536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00.000Z,3.3900,17850,United Kingdom,532942c37207951778ca5258c9b4de0cf4b789f947acdff1ebb7747857250548,initial,Online Retail.xlsx,Online Retail,3,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00006-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-315-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-10T20:53:49.431Z,20.3400,2010-12-01,2010,12,2026-08-10T20:53:49.431Z
f871cfdcba9b6cfe9fa0c049fc80f84ac65f914f8848efe4af499c6fdac859cb,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00.000Z,3.3900,17850,United Kingdom,5d2f7ca64405415988e613920f8bbd7b3dd24e7632def92f5a560dd284d8d55b,initial,Online Retail.xlsx,Online Retail,6,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00007-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-316-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-10T20:53:49.431Z,20.3400,2010-12-01,2010,12,2026-08-10T20:53:49.431Z
86a800eb6412b8679412adfaebea76891efe154342eaa161ce79ecb41cc1a55a,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00.000Z,3.3900,17850,United Kingdom,97792215273e2a88a600b1e0c97a2cc746f3dd545256ad9a76ab05e045495e91,initial,Online Retail.xlsx,Online Retail,5,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00008-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-317-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-10T20:53:49.431Z,20.3400,2010-12-01,2010,12,2026-08-10T20:53:49.431Z
ebb5304c4d44f54ef8cb0c618c67b7c0439fdcce9f5c05922ec85d1d4bdbb90c,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00.000Z,2.7500,17850,United Kingdom,c1cd806692dd9c0f64a38acc337200ebc4eff66727b38ab69269e4a98bd4744f,initial,Online Retail.xlsx,Online Retail,4,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00009-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-318-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z,v1,2026-08-10T20:53:49.431Z,22.0000,2010-12-01,2010,12,2026-08-10T20:53:49.431Z
28fe3ea9b4ccfebcde298f07fc2aa7b5f42aa534599720dd4bd24e3934a6b02e,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01T08:28:00.000Z,1.8500,17850,United Kingdom,746abb26f54dbc3cb3c049be1cc007123647a976c67af0902b366aff13acf3da,initial,Online Retail.xlsx,Online Retail,10,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00010-tid-762981857188285749-d78f

## 9. Persist the batch-specific Silver candidate

The candidate is written to its own batch-specific Delta path with overwrite mode. This makes reruns deterministic without overwriting another batch. The next notebook reads this exact path and merges the records into the final Silver table.

In [0]:
(
    silver_candidate_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_candidate_path)
)

persisted_candidate_df = spark.read.format("delta").load(silver_candidate_path)
persisted_candidate_count = persisted_candidate_df.count()
if persisted_candidate_count != valid_count:
    raise AssertionError(
        f"Persisted candidate count mismatch: expected {valid_count}, "
        f"found {persisted_candidate_count}."
    )

print(f"✅ Silver candidate written: {silver_candidate_path}")
print(f"Candidate rows: {persisted_candidate_count:,}")

✅ Silver candidate written: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/landing/silver_candidates/batch_id=initial
Candidate rows: 315,101


## 10. Upsert batch-level quality metrics

The metrics table is pre-created by `lab04_00_setup`. One row is maintained per `(batch_id, contract_version)` so reruns update the same reconciliation record rather than creating duplicates.


In [0]:
quality_metrics_df = (
    build_quality_metrics(quality_df)
    .withColumnRenamed("input_rows", "bronze_rows")
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("contract_version", F.lit(contract_version))
    .withColumn(
        "valid_percentage",
        F.round(
            F.col("valid_rows") / F.col("bronze_rows") * 100,
            4,
        ),
    )
    .withColumn(
        "rejected_percentage",
        F.round(
            F.col("rejected_rows") / F.col("bronze_rows") * 100,
            4,
        ),
    )
    .withColumn(
        "candidate_path",
        F.lit(silver_candidate_path),
    )
    .withColumn(
        "measured_at",
        F.current_timestamp(),
    )
)

if not spark.catalog.tableExists(quality_metrics_table):
    raise RuntimeError(
        f"Required quality-metrics target does not exist: "
        f"{quality_metrics_table}. "
        "Run lab04_00_setup manually before executing the Job."
    )

metrics_target_schema = {
    field.name: field.dataType.simpleString()
    for field in spark.table(quality_metrics_table).schema.fields
}

metrics_source_schema = {
    field.name: field.dataType.simpleString()
    for field in quality_metrics_df.schema.fields
}

if metrics_target_schema != metrics_source_schema:
    raise AssertionError(
        "Quality-metrics target schema does not match the generated metrics. "
        f"Target: {metrics_target_schema}; "
        f"Generated: {metrics_source_schema}"
    )

(
    DeltaTable.forName(spark, quality_metrics_table)
    .alias("target")
    .merge(
        quality_metrics_df.alias("source"),
        (
            "target.batch_id = source.batch_id "
            "AND target.contract_version = source.contract_version"
        ),
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

display(quality_metrics_df)


bronze_rows,valid_rows,rejected_rows,cancelled_rows,missing_customer_rows,duplicate_rows,batch_id,contract_version,valid_percentage,rejected_percentage,candidate_path,measured_at
433737,315101,118636,7430,108107,3427,initial,v1,72.6479,27.3521,/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/landing/silver_candidates/batch_id=initial,2026-08-10T20:53:59.430Z


## 11. Final validation and evidence

This final cell verifies the three persistent outputs for the selected batch: the candidate Delta path, quarantine table, and quality metrics table.

In [0]:
metrics_key_count = (
    spark.table(quality_metrics_table)
    .filter(
        (F.col("batch_id") == batch_id)
        & (F.col("contract_version") == contract_version)
    )
    .count()
)

if metrics_key_count != 1:
    raise AssertionError(f"Expected one quality-metrics row, found {metrics_key_count}.")

validation_df = spark.createDataFrame(
    [
        ("bronze_rows", bronze_batch_count),
        ("valid_candidate_rows", persisted_candidate_count),
        ("quarantined_rows", quarantine_batch_count),
        ("duplicate_rows_rejected", duplicate_row_count),
        ("quality_metric_rows_for_key", metrics_key_count),
    ],
    ["validation", "result"],
)

display(validation_df)
print("✅ Silver-quality processing completed successfully.")

validation,result
bronze_rows,433737
valid_candidate_rows,315101
quarantined_rows,118636
duplicate_rows_rejected,3427
quality_metric_rows_for_key,1


✅ Silver-quality processing completed successfully.


## 12. Completion checklist and next notebook

This notebook is complete when:

- every selected Bronze row is classified as `VALID` or `REJECTED`;
- the reconciliation check proves `Bronze = valid + rejected`;
- each rejected record has at least one explicit reason code;
- exact business duplicates are absent from the Silver candidate;
- quarantine and quality metrics are idempotent for the batch and contract;
- the batch-specific Silver candidate is readable from its Delta path.

Recommended screenshots: rule-failure summary, reconciliation counts, sample rejected rows with reason arrays, normalized candidate schema, and final validation table.

### Next notebook

Continue with **`lab04_04_silver_merge.ipynb`**. It will load the persisted candidate path, create the final Silver Delta table with a defined schema, perform an upsert with `MERGE`, and prove that rerunning the same batch does not change the row count.